# B8: our full TinyLlama model

**Goal:** trained weights -> our own complete forward -> HF-matching logits and greedy tokens in FP32.

```
IDs (N,) -> embedding (N,2048) -> 22 decoder layers
         -> final RMSNorm -> lm_head -> logits (N,32000)
```

HF supplies the tokenizer, weights and independent reference. Our forward uses only the modules we built; it does not call HF model code.

**Spec:** `model(input_ids, positions, AttentionMetadata)` returns logits `(N, vocab)` in FP32. Input IDs and positions are flattened `(N,)`; metadata positions must agree. Strict checkpoint loading, no KV cache.

Run top to bottom in **Colab on a T4**. Two FP32 model copies need about **8.8 GB for weights** plus temporary memory. Close previous GPU-heavy notebook kernels first.

In [1]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


c23abff B8: full Llama model, strict loading and greedy parity lesson



In [2]:
import importlib
import torch
import tinyserve.models.llama, tinyserve.probe
for module in (tinyserve.models.llama, tinyserve.probe):
    importlib.reload(module)
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.models.attention import AttentionMetadata
from tinyserve.basics import load_model, encode
from tinyserve.envinfo import collect_env, save_result
from tinyserve.probe import compare_full_model, compare_greedy_steps

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


{'python': '3.13.16', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'git_sha': 'c23abff', 'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'transformers': '5.21.0'}


## 1. Test first

Run the model-component tests B2-B8. B8 adds complete-stack HF parity, strict loading, tied/untied weights, final norm and LM head wiring, packed isolation, causality and exact greedy agreement with HF `generate(use_cache=False)` on a small model.

A failure stops the notebook. Read skips: HF, SDPA, plotting and CUDA tests should be available here.

Real full-stack FP32 logits use `rtol=1e-5, atol=1e-4` for accumulated rounding. Token equality is a separate required assertion. A synthetic FP16 test compares against the same rounded weights in FP32, not bit equality.

In [3]:
test_files = ["tests/test_b2_rmsnorm.py", "tests/test_b3_rope.py", "tests/test_b4_attention.py", "tests/test_b5_gqa.py",
              "tests/test_b6_mlp.py", "tests/test_b7_decoder.py", "tests/test_b8_model.py"]
test_run = subprocess.run([sys.executable, "-m", "pytest", *test_files, "-v", "-ra"], capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/tinyserve
configfile: pyproject.toml
plugins: anyio-4.15.1, platformdirs-4.13.6, langsmith-0.14.13, typeguard-4.6.0
collecting ... collected 143 items

tests/test_b2_rmsnorm.py::test_hand_computed_example PASSED              [  0%]
tests/test_b2_rmsnorm.py::test_matches_float64_formula PASSED            [  1%]
tests/test_b2_rmsnorm.py::test_fp32_matches_huggingface PASSED           [  2%]
tests/test_b2_rmsnorm.py::test_fp16_matches_huggingface_exactly PASSED   [  2%]
tests/test_b2_rmsnorm.py::test_fp16_close_to_fp32 PASSED                 [  3%]
tests/test_b2_rmsnorm.py::test_scale_invariant_without_eps PASSED        [  4%]
tests/test_b2_rmsnorm.py::test_unit_weight_gives_unit_rms PASSED         [  4%]
tests/test_b2_rmsnorm.py::test_does_not_subtract_the_mean PASSED         [  5%]
test

In [4]:
import psutil, subprocess
ram = psutil.virtual_memory()
print(f"RAM available: {ram.available / 2**30:.2f} GiB / {ram.total / 2**30:.2f} GiB")
print(subprocess.run(
["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total", "--format=csv"],
capture_output=True, text=True, check=True
).stdout)

RAM available: 11.10 GiB / 12.67 GiB
memory.used [MiB], memory.free [MiB], memory.total [MiB]
3 MiB, 14910 MiB, 15360 MiB



## 2. Strictly load all trained weights

The names match HF: `model.embed_tokens`, `model.layers.0...21`, `model.norm`, `lm_head`. `strict=True` rejects missing, extra or incorrectly shaped weights. This copies weights into **our architecture**, not an HF wrapper.

TinyLlama does not tie its input/output embeddings: there are two separate 32000-by-2048 matrices. Their jobs differ: lookup a token's initial vector, versus score every possible next token.

In [5]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
our_params = sum(parameter.numel() for parameter in our_model.parameters())
hf_params = sum(parameter.numel() for parameter in hf_model.parameters())
assert our_params == hf_params
assert len(our_model.model.layers) == 22
print(f"parameters: ours {our_params:,}, HF {hf_params:,}")
print("weight tensors:", len(our_model.state_dict()))
print("tied embeddings:", our_model.lm_head.weight is our_model.model.embed_tokens.weight)


parameters: ours 1,100,048,384, HF 1,100,048,384
weight tensors: 201
tied embeddings: False


## 3. Full logits on multiple prompts

B7 used HF's input for each isolated layer. Here our layer 0 feeds our layer 1, and so on through all 22 layers, the final norm and LM head.

Compare **every logits row** and its argmax ID. A numerical tolerance pass alone is insufficient: two nearly tied candidates can switch places. Record the top-two margin on the last row to see how close the candidates are.

In [6]:
prompts = ["The capital of France is", "The capital of Germany is", "A tiny language model can"]
prompt_rows = []
for prompt_text in prompts:
    prompt_ids = encode(tokenizer, prompt_text, DEVICE)
    result = compare_full_model(our_model, hf_model, prompt_ids)
    result["prompt"] = prompt_text
    prompt_rows.append(result)
    print(f"{prompt_text!r}: max |diff| {result['max_abs_diff']:.2e}, "
          f"all argmax equal {result['argmax_equal']}, next {tokenizer.decode([result['next_token_id']])!r}, "
          f"HF top-two margin {result['hf_top_two_margin']:.4f}")


'The capital of France is': max |diff| 9.54e-06, all argmax equal True, next 'Paris', HF top-two margin 1.1198
'The capital of Germany is': max |diff| 9.54e-06, all argmax equal True, next 'Berlin', HF top-two margin 1.2596
'A tiny language model can': max |diff| 9.06e-06, all argmax equal True, next 'be', HF top-two margin 0.5775


## 4. Flattened packed requests

Concatenate two prompts, restart positions per request and assign distinct sequence IDs. Run one flattened forward, then compare with independent forwards. The model must not let one request influence another.

In [7]:
first_ids = encode(tokenizer, prompts[0], DEVICE)
second_ids = encode(tokenizer, prompts[1], DEVICE)
packed_ids = torch.cat([first_ids, second_ids])
first_positions = torch.arange(first_ids.numel(), device=DEVICE)
second_positions = torch.arange(second_ids.numel(), device=DEVICE)
packed_positions = torch.cat([first_positions, second_positions])
packed_sequence_ids = torch.cat([torch.zeros_like(first_ids), torch.ones_like(second_ids)])
with torch.no_grad():
    packed_logits = our_model(packed_ids, packed_positions, AttentionMetadata(packed_positions, packed_sequence_ids))
    separate_logits = torch.cat([
        our_model(first_ids, first_positions, AttentionMetadata(first_positions)),
        our_model(second_ids, second_positions, AttentionMetadata(second_positions)),
    ])
packed_max_diff = (packed_logits - separate_logits).abs().max().item()
torch.testing.assert_close(packed_logits, separate_logits, rtol=1e-5, atol=1e-4)
assert torch.equal(packed_logits.argmax(-1), separate_logits.argmax(-1))
print("packed vs separate max difference:", packed_max_diff)


packed vs separate max difference: 4.57763671875e-05


## 5. Token-exact greedy parity, step by step

For each prompt, run both models on the same growing prefix. Check logits and argmax at every step, append the agreed token, and stop on EOS or 12 new tokens. A mismatch raises immediately.

This is a **verification loop**, not the public generation API (C2). No KV cache, no sampling, no speed claims. Raw text prompts can continue into lists instead of giving concise chat answers; we care about matching HF, not the prose style.

In [8]:
greedy_rows = []
for prompt_text in prompts:
    prompt_ids = encode(tokenizer, prompt_text, DEVICE)
    generated_ids, step_rows = compare_greedy_steps(our_model, hf_model, prompt_ids, max_new_tokens=12)
    generated_text = tokenizer.decode(generated_ids.tolist(), skip_special_tokens=True)
    greedy_rows.append({"prompt": prompt_text, "token_ids": generated_ids.tolist(), "text": generated_text,
                        "new_tokens": len(step_rows), "steps": step_rows})
    print(f"{len(step_rows)} matching steps: {generated_text!r}")


12 matching steps: 'The capital of France is Paris.\n\n2. B. The capital of Germany'
12 matching steps: 'The capital of Germany is Berlin.\n\n2. The capital of the United States'
12 matching steps: 'A tiny language model can be trained to predict the next word in a sentence. This'


## 6. Save results and three interview questions

1. **Why full-stack parity?** It checks our own layer outputs feeding later layers, not isolated blocks on HF inputs.
2. **Why logits tolerance plus exact tokens?** Small score errors can change argmax near a tie.
3. **What does HF still do?** Supply trained weights/tokenizer and the test oracle. Our forward computes the entire model.

Error ranges below are across prompts and greedy steps, not performance trials. Save the notebook with outputs and retain this JSON.

**Next: C1 sampling** (greedy, temperature, top-k, top-p), then C2 generation and C3 KV cache.

In [11]:
all_differences = [row["max_abs_diff"] for row in prompt_rows]
all_differences += [step["max_abs_diff"] for row in greedy_rows for step in row["steps"]]
saved_path = save_result("B8_full_model", {
    "dtype": "float32", "backend": "reference", "hf_backend": "eager", "use_cache": False,
    "num_params": our_params,
    "prompt_comparison": prompt_rows,
    "packed_max_abs_diff": packed_max_diff,
    "greedy_comparison": greedy_rows,
    "diff_range_across_prompts_and_steps": {"min": min(all_differences), "max": max(all_differences)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


/content/tinyserve/results/B8_full_model.json
{
  "name": "B8_full_model",
  "timestamp_utc": "2026-10-31T06:39:17Z",
  "env": {
    "python": "3.13.16",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "git_sha": "c23abff",
    "gpu": "Tesla T4",
    "torch": "2.11.0+cu130",
    "cuda": "13.0",
    "transformers": "5.21.0"
  },
  "data": {
    "dtype": "float32",
    "backend": "reference",
    "hf_backend": "eager",
    "use_cache": false,
    "num_params": 1100048384,
    "prompt_comparison": [
      {
        "tokens": 6,
        "max_abs_diff": 9.5367431640625e-06,
        "argmax_equal": true,
        "next_token_id": 3681,
        "hf_top_two_margin": 1.1197996139526367,
        "prompt": "The capital of France is"
      },
      {
        "tokens": 6,
        "max_abs_diff": 9.5367431640625e-06,
        "argmax_equal": true,
        "next_token_id": 5115,
        "hf_top_two_margin": 1.259572982788086,
        "prompt": "The capital of Germany is"
      },
      {
  

In [12]:
from pathlib import Path
print(Path("/content/tinyserve/results/B8_full_model.json").read_text())

{
  "name": "B8_full_model",
  "timestamp_utc": "2026-10-31T06:39:17Z",
  "env": {
    "python": "3.13.16",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "git_sha": "c23abff",
    "gpu": "Tesla T4",
    "torch": "2.11.0+cu130",
    "cuda": "13.0",
    "transformers": "5.21.0"
  },
  "data": {
    "dtype": "float32",
    "backend": "reference",
    "hf_backend": "eager",
    "use_cache": false,
    "num_params": 1100048384,
    "prompt_comparison": [
      {
        "tokens": 6,
        "max_abs_diff": 9.5367431640625e-06,
        "argmax_equal": true,
        "next_token_id": 3681,
        "hf_top_two_margin": 1.1197996139526367,
        "prompt": "The capital of France is"
      },
      {
        "tokens": 6,
        "max_abs_diff": 9.5367431640625e-06,
        "argmax_equal": true,
        "next_token_id": 5115,
        "hf_top_two_margin": 1.259572982788086,
        "prompt": "The capital of Germany is"
      },
      {
        "tokens": 6,
        "max_abs_diff": 9.0

In [10]:
import gc
del our_model, hf_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")


GPU memory still held by this kernel: 11 MiB
